# P7513 clustering Squidpy interactive plot

This notebook recreates a publication-style version of the `clustering_squidpy` spatial scatter and UMAP outputs for P7513 MERSCOPE.

It loads the pregenerated clustered AnnData object, then plots the spatial leiden scatter on the left and the leiden-only UMAP on the right. Edit `CLUSTER_STYLE` below to rename leiden clusters and assign your own colors.

## Environment

Run this notebook from the MerXen conda environment so `anndata`, `matplotlib`, and the local `merxen` package are available.

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.plotting import save_figure


In [ ]:
PAIR_ID = "P7513"
PLATFORM = "MERSCOPE"
PLATFORM_DIR = PLATFORM.lower()

CLUSTERED_H5AD_PATH = (
    REPO_ROOT
    / "results"
    / PAIR_ID
    / "clustering_squidpy"
    / "clustering_squidpy_out"
    / PLATFORM_DIR
    / f"{PAIR_ID}_{PLATFORM}_clustered.h5ad"
)
OUTPUT_DIR = REPO_ROOT / "results" / PAIR_ID / "clustering_squidpy" / "clustering_squidpy_out" / PLATFORM_DIR
OUTPUT_PATH = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_spatial_umap_leiden_interactive.png"
OUTPUT_PDF_PATH = OUTPUT_PATH.with_suffix(".pdf")

print(f"Clustered AnnData: {CLUSTERED_H5AD_PATH}")
print(f"PNG output:        {OUTPUT_PATH}")
print(f"PDF output:        {OUTPUT_PDF_PATH}")


In [ ]:
adata = ad.read_h5ad(CLUSTERED_H5AD_PATH)

print(adata)
print(f"spatial coordinates: {'spatial' in adata.obsm}")
print(f"UMAP coordinates:    {'X_umap' in adata.obsm}")
print(f"leiden clusters:     {adata.obs['leiden'].nunique() if 'leiden' in adata.obs else 'missing'}")


In [ ]:
def _category_sort_key(value: object) -> tuple[int, float | str]:
    text = str(value)
    try:
        return (0, float(text))
    except ValueError:
        return (1, text)


def _cluster_categories(adata_obj: ad.AnnData, cluster_key: str = "leiden") -> list[str]:
    if cluster_key not in adata_obj.obs:
        raise KeyError(f"Expected adata.obs[{cluster_key!r}] for cluster labels.")
    labels = adata_obj.obs[cluster_key]
    if isinstance(labels.dtype, pd.CategoricalDtype):
        categories = [str(category) for category in labels.cat.categories]
    else:
        categories = [str(category) for category in pd.unique(labels.astype(str))]
    return sorted(categories, key=_category_sort_key)


def _default_cluster_colors(adata_obj: ad.AnnData, categories: list[str], cluster_key: str = "leiden") -> dict[str, str]:
    stored_colors = list(adata_obj.uns.get(f"{cluster_key}_colors", []))
    if len(stored_colors) >= len(categories):
        return {category: stored_colors[index] for index, category in enumerate(categories)}

    cmap = plt.get_cmap("tab20")
    return {
        category: cmap(index % cmap.N)
        for index, category in enumerate(categories)
    }


def build_default_cluster_style(
    adata_obj: ad.AnnData,
    cluster_key: str = "leiden",
) -> dict[str, dict[str, str]]:
    """Create an editable style dict from the existing leiden categories and colors."""
    categories = _cluster_categories(adata_obj, cluster_key=cluster_key)
    colors = _default_cluster_colors(adata_obj, categories, cluster_key=cluster_key)
    return {
        category: {"name": f"Cluster {category}", "color": colors[category]}
        for category in categories
    }


CLUSTER_STYLE = build_default_cluster_style(adata, cluster_key="leiden")
CLUSTER_STYLE


## Edit Cluster Names And Colors

Edit the dictionary below to map leiden IDs to the cell type names and colors you want in the figure. Any clusters you leave out will fall back to `Cluster <id>` and the stored Scanpy color.

To merge multiple leiden clusters visually, give them the same `name` and `color`. They will be plotted with one shared color and collapsed to one legend entry.

In [ ]:
# Replace the names below with your preferred annotations.
# Colors use Matplotlib CSS4 color names, e.g. from matplotlib.colors.CSS4_COLORS.
#
# To merge clusters visually, assign the same name and color to multiple leiden IDs.
# The legend will collapse repeated names into one entry.
CLUSTER_STYLE.update({
    "0": {"name": "Fibroblasts", "color": "sienna"},
    "1": {"name": "Vascular cells", "color": "teal"},
    "2": {"name": "Microglia", "color": "goldenrod"},
    "3": {"name": "Vascular cells", "color": "teal"},
    "4": {"name": "Astrocytes", "color": "forestgreen"},
    "5": {"name": "Excitatory neurons", "color": "darkorange"},
    "6": {"name": "Inhibitory neurons", "color": "crimson"},
    "7": {"name": "Excitatory neurons", "color": "darkorange"},
    "8": {"name": "Oligodendrocytes", "color": "royalblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "plum"},
    "10": {"name": "Astrocytes", "color": "forestgreen"},
})
CLUSTER_STYLE


In [ ]:
def _normalize_cluster_style(
    adata_obj: ad.AnnData,
    cluster_style: dict[str, object] | None,
    cluster_key: str,
) -> dict[str, dict[str, str]]:
    categories = _cluster_categories(adata_obj, cluster_key=cluster_key)
    defaults = build_default_cluster_style(adata_obj, cluster_key=cluster_key)
    if cluster_style is None:
        return defaults

    normalized: dict[str, dict[str, str]] = {}
    for category in categories:
        style_value = cluster_style.get(category, defaults[category])
        if isinstance(style_value, str):
            normalized[category] = {"name": defaults[category]["name"], "color": style_value}
        elif isinstance(style_value, tuple | list) and len(style_value) == 2:
            normalized[category] = {"name": str(style_value[0]), "color": style_value[1]}
        elif isinstance(style_value, dict):
            normalized[category] = {
                "name": str(style_value.get("name", defaults[category]["name"])),
                "color": style_value.get("color", defaults[category]["color"]),
            }
        else:
            raise TypeError(
                "Cluster style values must be colors, (name, color) pairs, or "
                f"dicts with name/color keys. Got {type(style_value)!r} for cluster {category!r}."
            )
    return normalized


def _plot_cluster_points(
    ax: plt.Axes,
    coords: np.ndarray,
    labels: pd.Series,
    cluster_style: dict[str, dict[str, str]],
    *,
    point_size: float,
    alpha: float,
    add_legend: bool,
    legend_marker_size: float = 9.0,
    rasterized: bool = True,
) -> list[Line2D]:
    legend_handles: list[Line2D] = []
    legend_names_seen: set[str] = set()
    label_text = labels.astype(str)

    for cluster_id, style in cluster_style.items():
        mask = label_text.to_numpy() == cluster_id
        if not np.any(mask):
            continue
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            alpha=alpha,
            c=[style["color"]],
            edgecolors="none",
            linewidths=0,
            rasterized=rasterized,
        )
        if add_legend and style["name"] not in legend_names_seen:
            legend_names_seen.add(style["name"])
            legend_handles.append(
                Line2D(
                    [0],
                    [0],
                    marker="o",
                    linestyle="",
                    label=style["name"],
                    markerfacecolor=style["color"],
                    markeredgecolor="none",
                    markersize=legend_marker_size,
                    alpha=1.0,
                )
            )
    return legend_handles


def plot_spatial_umap_leiden_interactive(
    adata_obj: ad.AnnData,
    output_path: Path | str | None = None,
    *,
    cluster_style: dict[str, object] | None = None,
    cluster_key: str = "leiden",
    spatial_point_size: float = 0.45,
    spatial_alpha: float = 0.65,
    umap_point_size: float = 0.45,
    umap_alpha: float = 0.65,
    legend_marker_size: float = 9.0,
    legend_fontsize: float = 10.0,
    legend_loc: str = "center left",
    legend_bbox_to_anchor: tuple[float, float] | None = (1.02, 0.5),
    figsize: tuple[float, float] = (12.0, 5.8),
    wspace: float = 0.05,
    dpi: int = 220,
) -> plt.Figure:
    """Plot spatial leiden labels and leiden-only UMAP with shared cluster colors."""
    if "spatial" not in adata_obj.obsm:
        raise KeyError("Expected adata.obsm['spatial'] for the spatial panel.")
    if "X_umap" not in adata_obj.obsm:
        raise KeyError("Expected adata.obsm['X_umap'] for the UMAP panel.")

    style = _normalize_cluster_style(adata_obj, cluster_style, cluster_key)
    labels = adata_obj.obs[cluster_key]
    spatial_xy = np.asarray(adata_obj.obsm["spatial"], dtype=float)
    umap_xy = np.asarray(adata_obj.obsm["X_umap"], dtype=float)

    fig, (spatial_ax, umap_ax) = plt.subplots(1, 2, figsize=figsize)

    _plot_cluster_points(
        spatial_ax,
        spatial_xy,
        labels,
        style,
        point_size=spatial_point_size,
        alpha=spatial_alpha,
        add_legend=False,
    )
    spatial_ax.set_aspect("equal", adjustable="box")
    spatial_ax.invert_yaxis()
    spatial_ax.set_title("")
    spatial_ax.set_xlabel("")
    spatial_ax.set_ylabel("")
    spatial_ax.set_xticks([])
    spatial_ax.set_yticks([])
    spatial_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in spatial_ax.spines.values():
        spine.set_visible(False)
    spatial_ax.set_frame_on(False)

    legend_handles = _plot_cluster_points(
        umap_ax,
        umap_xy,
        labels,
        style,
        point_size=umap_point_size,
        alpha=umap_alpha,
        add_legend=True,
        legend_marker_size=legend_marker_size,
    )
    umap_ax.set_title("")
    umap_ax.set_xlabel("UMAP1")
    umap_ax.set_ylabel("UMAP2")
    umap_ax.set_xticks([])
    umap_ax.set_yticks([])
    umap_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in umap_ax.spines.values():
        spine.set_visible(False)
    umap_ax.set_frame_on(False)

    if legend_handles:
        legend_kwargs = {
            "handles": legend_handles,
            "loc": legend_loc,
            "frameon": False,
            "fontsize": legend_fontsize,
        }
        if legend_bbox_to_anchor is not None:
            legend_kwargs["bbox_to_anchor"] = legend_bbox_to_anchor
        umap_ax.legend(**legend_kwargs)

    fig.subplots_adjust(wspace=wspace)
    if output_path is not None:
        save_figure(fig, output_path, dpi=dpi, bbox_inches="tight")
    return fig


In [ ]:
# Optional: save the interactive output.
# save_figure() is the repo helper used by production plots; it writes PNG plus a sibling PDF.

CLUSTER_STYLE = {
    "0": {"name": "Fibroblasts", "color": "sienna"},
    "1": {"name": "Vascular cells", "color": "teal"},
    "2": {"name": "Microglia", "color": "goldenrod"},
    "3": {"name": "Vascular cells", "color": "teal"},
    "4": {"name": "Astrocytes", "color": "forestgreen"},
    "5": {"name": "Excitatory neurons", "color": "darkorange"},
    "6": {"name": "Inhibitory neurons", "color": "crimson"},
    "7": {"name": "Excitatory neurons", "color": "darkorange"},
    "8": {"name": "Oligodendrocytes", "color": "royalblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "plum"},
    "10": {"name": "Astrocytes", "color": "forestgreen"},
}

CLUSTER_STYLE = {
    "0": {"name": "Fibroblasts", "color": "red"},
    "1": {"name": "Vascular cells", "color": "purple"},
    "2": {"name": "Microglia", "color": "gold"},
    "3": {"name": "Vascular cells", "color": "darkcyan"},
    "4": {"name": "Astrocytes", "color": "limegreen"},
    "5": {"name": "Excitatory neurons", "color": "orangered"},
    "6": {"name": "Inhibitory neurons", "color": "darkcyan"},
    "7": {"name": "Excitatory neurons", "color": "orangered"},
    "8": {"name": "Oligodendrocytes", "color": "dodgerblue"},
    "9": {"name": "Oligodendrocyte precursors", "color": "hotpink"},
    "10": {"name": "Astrocytes", "color": "limegreen"},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig = plot_spatial_umap_leiden_interactive(
    adata,
    output_path=OUTPUT_PATH,
    cluster_style=CLUSTER_STYLE,
    spatial_point_size=0.45,
    spatial_alpha=0.65,
    umap_point_size=0.45,
    umap_alpha=0.65,
    legend_marker_size=9.0,s
    legend_fontsize=10,
    figsize=(12.0, 5.8),
    wspace=0.05,
)
print(f"Saved PNG: {OUTPUT_PATH}")
print(f"Saved PDF: {OUTPUT_PDF_PATH}")
